# Phase 4a — Golden Relevance-Judgement Set

Builds the frozen query -> relevant-chunk set used to score **all three** retrievers
(BM25 here, dense in Phase 5, hybrid in Phase 6). Strategy = **Option C**
(LLM-seeded, manually curated); see `notes/phase_04_retrieval_decisions.md`.

**Execution has a deliberate manual break:**
1. Generate candidate queries from a stratified seed sample (LLM, ~2c on gpt-4o-mini).
2. **STOP. Curate `golden_candidates.csv` by hand** — paraphrase each query to remove
   lexical echo of its source chunk (this is the bias mitigation, not optional),
   keep/drop, confirm relevant chunks.
3. Load the curated CSV, validate, freeze to `golden_queries.parquet`, record the hash.

> Reproducibility lock: EDGAR year = **2020**, seed = **42**, chunk config 1500/200.
> The same `golden_queries.parquet` is reused unchanged at n=500 scale-up — the new
> filings only add distractors; the query set and relevant ids never change.

In [1]:
%load_ext autoreload
%autoreload 2

import sys, json, hashlib
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.retrieval.golden_dataset_builder import (
    sample_seed_chunks, build_candidate_set, save_candidate_set,
    load_curated_golden_set, save_golden_set, load_golden_set, hash_golden_set,
    make_openai_client, SEED, DEFAULT_GEN_MODEL,
)

CHUNKS_PATH    = ROOT / "data" / "processed" / "chunks_n200.parquet"
CANDIDATES_CSV = ROOT / "data" / "processed" / "golden_candidates.csv"
GOLDEN_PATH    = ROOT / "data" / "processed" / "golden_queries.parquet"
print("chunks:", CHUNKS_PATH)

chunks: d:\General IT\AI-ML-LJMU\final_thesis\code\data\processed\chunks_n200.parquet


## 1. Load chunks + draw the stratified seed sample (deterministic, seed 42)

In [2]:
chunks = pd.read_parquet(CHUNKS_PATH)
print(f"{len(chunks):,} chunks | sources: {chunks['source'].value_counts().to_dict()}")

# ~30 EDGAR + ~30 earnings, stratified across subtype. Tune n_* if you want a
# larger set, but 60 is the right scope for a solo thesis (see decisions note).
seeds = sample_seed_chunks(chunks, n_edgar=30, n_earnings=30, seed=SEED)
print(f"seed chunks: {len(seeds)}")
print(seeds.groupby(['source','subtype']).size())

21,050 chunks | sources: {'edgar': 16967, 'earnings': 4083}
seed chunks: 60
source    subtype         
earnings  full                11
          prepared_remarks     7
          qa                  12
edgar     section_1            6
          section_1A          10
          section_7            6
          section_8            8
dtype: int64


## 2. Generate candidate queries (LLM)

Set `OPENAI_API_KEY` in your environment first (PowerShell: `$env:OPENAI_API_KEY="sk-..."`).
Cost: ~60 calls on `gpt-4o-mini` ≈ a couple of cents.

In [5]:
import os
from dotenv import load_dotenv
load_dotenv(override=True) 

assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY before running this cell."

client = make_openai_client()  # reads OPENAI_API_KEY
candidates = build_candidate_set(
    seeds, chunks, client=client, model=DEFAULT_GEN_MODEL,
    n_questions=1, seed=SEED, sibling_window=1,
)
save_candidate_set(candidates, CANDIDATES_CSV)
print(f"Wrote {len(candidates)} candidate queries -> {CANDIDATES_CSV}")
candidates.head(3)

Wrote 60 candidate queries -> d:\General IT\AI-ML-LJMU\final_thesis\code\data\processed\golden_candidates.csv


,query_id,keep,query_text_raw,query_text_curated,source,subtype,seed_chunk_id,relevant_chunk_ids,candidate_siblings,notes
0,q000,1,What closing remarks did Michael Petters make ...,,earnings,qa,earnings_0000_qa_chunk_030,earnings_0000_qa_chunk_030,earnings_0000_qa_chunk_029,
1,q001,1,What is the expected range of productivity con...,,earnings,qa,earnings_0002_qa_chunk_012,earnings_0002_qa_chunk_012,earnings_0002_qa_chunk_011|earnings_0002_qa_ch...,
2,q002,1,What was the impact of promotions on sales and...,,earnings,full,earnings_0006_full_chunk_020,earnings_0006_full_chunk_020,earnings_0006_full_chunk_019|earnings_0006_ful...,


In [6]:
# Build a LLM-friendly file: candidate query + its seed chunk text, query_id-keyed.
cand = pd.read_csv(CANDIDATES_CSV, encoding="utf-8-sig")
seed_text = chunks.set_index("chunk_id")["text"]
enriched = cand[["query_id", "source", "subtype", "seed_chunk_id", "query_text_raw"]].copy()
enriched["seed_chunk_text"] = enriched["seed_chunk_id"].map(seed_text).fillna("")
# trim chunk text so the file stays manageable for a chat upload
enriched["seed_chunk_text"] = enriched["seed_chunk_text"].str.slice(0, 1200)
ENRICHED_CSV = ROOT / "data" / "processed" / "golden_candidates_for_paraphrase.csv"
enriched.to_csv(ENRICHED_CSV, index=False, encoding="utf-8-sig")
print("wrote", ENRICHED_CSV)

wrote d:\General IT\AI-ML-LJMU\final_thesis\code\data\processed\golden_candidates_for_paraphrase.csv


In [8]:
para = pd.read_csv("gemini_output.csv")  # query_id, query_text_curated
cand = pd.read_csv(CANDIDATES_CSV, encoding="utf-8-sig")
cand = cand.drop(columns=["query_text_curated"]).merge(para, on="query_id", how="left")
# reorder so it matches the expected curation schema, then save
cand.to_csv(CANDIDATES_CSV, index=False, encoding="utf-8-sig")

In [10]:
# --- Apply the 7 Bucket-2 query fixes (over-generalised -> anchor restored) ---
# Reads the pristine backup, patches only 7 rows by query_id, writes the
# processed file in place. Idempotent: safe to re-run. Numbers/years/standard
# accounting terms are kept (not chunk-echo); only proper nouns stay removed.
import shutil

ARCHIVED_CSV = ROOT / "data" / "processed" / "archived" / "golden_candidates.csv"
# fall back to a sibling 'archived' if you nested it elsewhere
if not ARCHIVED_CSV.exists():
    ARCHIVED_CSV = ROOT / "archived" / "golden_candidates.csv"
assert ARCHIVED_CSV.exists(), f"backup not found at {ARCHIVED_CSV} — point this at your archived copy"

FIXES = {
    "q008": "What was the transaction banking revenue figure discussed for the 2022 fiscal year?",
    "q013": "What free cash flow conversion percentage target was referenced for the prior year?",
    "q015": "What was the purchase accounting amount reported for the second quarter?",
    "q034": "How many shares were issued to employees under the discounted stock purchase plan during 2020?",
    "q035": "What deferred acquisition cost amounts were amortised across the three most recent fiscal years?",
    "q043": "What interest rate applied to the convertible note issued in early 2020?",
    "q048": "How much did net sales grow in fiscal year 2020 compared with 2019?",
}

_df = pd.read_csv(ARCHIVED_CSV, encoding="utf-8-sig")
assert {"query_id", "query_text_curated"} <= set(_df.columns), f"unexpected cols: {list(_df.columns)}"

_applied = []
for qid, new_text in FIXES.items():
    m = _df["query_id"] == qid
    if not m.any():
        print(f"  WARN: {qid} not found")
        continue
    _applied.append((qid, str(_df.loc[m, "query_text_curated"].iloc[0]).strip(), new_text))
    _df.loc[m, "query_text_curated"] = new_text

_df.to_csv(CANDIDATES_CSV, index=False, encoding="utf-8-sig")  # processed/golden_candidates.csv

print(f"Patched {len(_applied)} rows; wrote {len(_df)} rows -> {CANDIDATES_CSV}")
for qid, old, new in _applied:
    print(f"\n[{qid}]\n  old: {old}\n  new: {new}")

Patched 7 rows; wrote 60 rows -> d:\General IT\AI-ML-LJMU\final_thesis\code\data\processed\golden_candidates.csv

[q008]
  old: What was the total income generated by the commercial banking platform during the prior calendar year?
  new: What was the transaction banking revenue figure discussed for the 2022 fiscal year?

[q013]
  old: What specific target for converting earnings into available cash was asked about during the Q&A session?
  new: What free cash flow conversion percentage target was referenced for the prior year?

[q015]
  old: What was the total value of accounting adjustments related to recent acquisitions during the second three-month period?
  new: What was the purchase accounting amount reported for the second quarter?

[q034]
  old: How many equity shares were distributed to employees through the discounted purchasing plan during the most recent calendar year?
  new: How many shares were issued to employees under the discounted stock purchase plan during 2020?

[q03

## 3. ⛔ MANUAL CURATION STEP (do this outside the notebook)

Open `data/processed/golden_candidates.csv` (Excel / a text editor) and for **each row**:

1. **`query_text_curated`** — rewrite `query_text_raw` in your own words so it does
   **not** reuse the source chunk's distinctive vocabulary. *This is the step that
   keeps the BM25-vs-dense comparison fair.* Empty curated text = the loader rejects it.
2. **`keep`** — set to `0` to drop a weak / unanswerable / duplicate query.
3. **`relevant_chunk_ids`** — pipe-separated. Pre-filled with the seed chunk. If a
   neighbour in **`candidate_siblings`** genuinely also answers the query, append it
   here (e.g. `e0_chunk_000|e0_chunk_001`). Multi-positive queries make Precision@K
   meaningful.
4. **`notes`** — optional.

Aim for ~50–60 kept queries, roughly balanced EDGAR/earnings. Save the CSV, then run on.

## 4. Load + validate the curated set, freeze, and hash (Only run this as the new fix is rolled out with recurate_golden_candidates.py to paraphrase the 60 queries second time)

In [14]:
golden = load_curated_golden_set(CANDIDATES_CSV, chunks, seed=SEED, require_curated_text=True)
print(f"validated golden queries: {len(golden)}")
print("arm balance:", golden['source'].value_counts().to_dict())
print("multi-positive queries:", int((golden['relevant_chunk_ids'].apply(len) > 1).sum()))

golden_hash = save_golden_set(golden, GOLDEN_PATH)
print("saved:", GOLDEN_PATH)
print("SHA-256:", golden_hash)

# round-trip check
assert hash_golden_set(load_golden_set(GOLDEN_PATH)) == golden_hash
print("round-trip hash verified ✓  (record this hash in phase_04_summary.json)")

validated golden queries: 60
arm balance: {'earnings': 30, 'edgar': 30}
multi-positive queries: 0
saved: d:\General IT\AI-ML-LJMU\final_thesis\code\data\processed\golden_queries.parquet
SHA-256: 325634f065147a49d6fba6e0fb021107d536b1aa717bcbbb4a10b68b93c0e72e
round-trip hash verified ✓  (record this hash in phase_04_summary.json)


    - Old Hash - SHA-256: 92acb35b6d0af673b60715ac2507a33ca1f183414849f77db347d57412873aa9


- validated golden queries: 60
- arm balance: {'earnings': 30, 'edgar': 30}
- multi-positive queries: 0
- saved: d:\General IT\AI-ML-LJMU\final_thesis\code\data\processed\golden_queries.parquet
- SHA-256: 325634f065147a49d6fba6e0fb021107d536b1aa717bcbbb4a10b68b93c0e72e
- round-trip hash verified ✓  (record this hash in phase_04_summary.json)

In [15]:
# Sanity peek
golden[['query_id','source','subtype','query_text','relevant_chunk_ids']].head(8)

,query_id,source,subtype,query_text,relevant_chunk_ids
0,q000,earnings,qa,"In the closing remarks of the call, what did t...",[earnings_0000_qa_chunk_030]
1,q001,earnings,qa,What productivity contribution percentage was ...,[earnings_0002_qa_chunk_012]
2,q002,earnings,full,How did management describe the effect of prom...,[earnings_0006_full_chunk_020]
3,q003,earnings,prepared_remarks,What was the year-on-year growth rate of netwo...,[earnings_0013_prepared_chunk_004]
4,q004,earnings,full,How did management describe its capital deploy...,[earnings_0016_full_chunk_001]
5,q005,earnings,full,Did the company see any slowdown in its hydrau...,[earnings_0016_full_chunk_030]
6,q006,earnings,full,What dollar range was given for the permanent ...,[earnings_0018_full_chunk_020]
7,q007,earnings,qa,How many weeks late was potato crop planting i...,[earnings_0026_qa_chunk_007]
